# 0.1 Overview

The representative population pool is generated independently from multiple demographic datasets.

Although each attribute is statistically plausible, some attribute combinations are logically inconsistent.

This notebook applies a series of logical constraints to improve internal consistency before constructing behaviour profiles.

Input dataset:

- Representative Population Pool

Output:

- Representative Population Pool with Constraint Handlings

### 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

### 2. Set Up File Paths

In [2]:
from pathlib import Path

project_root = Path.cwd().parents[1]

input_path = (
    project_root
    / "output"
    / "demographic_profiles"
)

output_path = (
    project_root
    / "output"
    / "demographic_profiles"
)

output_path.mkdir(
    parents=True,
    exist_ok=True
)

### 3. Load Dataset

In [3]:
population_pool_file = (
    input_path
    / "representative_population_pool.parquet"
)

population_pool = pd.read_parquet(
    population_pool_file
)

display(population_pool.head())

print(population_pool.shape)

,Age,Gender,Household Type,Household Count,Economic Activity Status,Economic Count,Industry,Industry Count,Ethnic Group,Ethnic Count,Distance travelled to work,Travel Distance Count,Car Ownership,Car Ownership Count
0,0-4,Female,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,10km to less than 30km,0,No cars or vans in household,792
1,0-4,Female,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,10km to less than 30km,0,No cars or vans in household,221
2,0-4,Female,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,10km to less than 30km,0,1 car or van in household,2627
3,0-4,Female,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,10km to less than 30km,0,1 car or van in household,512
4,0-4,Female,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,10km to less than 30km,0,2 cars or vans in household,2614


(21450240, 14)


### 4. Age-related Constraints

The representative population pool is generated by combining multiple
demographic attributes into representative profiles. As a result,
some age groups may be paired with economic activity categories that
are not logically applicable.

Children aged **0–4** and **5–11** are not considered part of the
economically active population. Therefore, their economic activity
status is assigned as **"Does not apply"** to maintain logical
consistency within the representative population.

This constraint is applied before the employment-related constraints,
allowing employment-dependent attributes (e.g., industry and commuting
distance) to be updated consistently in subsequent processing.

In [4]:
YOUNG_CHILDREN = ["0-4", "5-11"]

AGE_MASK = (
    population_pool["Age"].isin(YOUNG_CHILDREN)
)

print("Economic Activity before applying age-related constraints")

display(
    population_pool.loc[
        AGE_MASK,
        "Economic Activity Status"
    ].value_counts()
)

Economic Activity before applying age-related constraints


Economic Activity Status
Does not apply                                                                                                                                                        680960
Economically active and a full-time student: In employment                                                                                                            680960
Economically inactive (excluding full-time students)                                                                                                                  680960
Economically active and a full-time student: Unemployed: Seeking work or waiting to start a job already obtained: Available to start working within 2 weeks           680960
Economically inactive and a full-time student                                                                                                                         680960
Economically active (excluding full-time students): In employment                                             

### 4.1 Before

In [5]:
age_inconsistent = (
    AGE_MASK &
    (
        population_pool["Economic Activity Status"]
        != "Does not apply"
    )
)

print(
    f"Inconsistent economic activity profiles identified: "
    f"{age_inconsistent.sum():,}"
)

Inconsistent economic activity profiles identified: 4,085,760


### 4.2 Apply Constraint

In [6]:
population_pool.loc[
    AGE_MASK,
    "Economic Activity Status"
] = "Does not apply"

### 4.3 After Applying Constraint

In [7]:
print("Economic Activity after applying age-related constraints")

display(
    population_pool.loc[
        AGE_MASK,
        "Economic Activity Status"
    ].value_counts()
)

Economic Activity after applying age-related constraints


Economic Activity Status
Does not apply    4766720
Name: count, dtype: int64

### 5. Employment-related Constraints

Employment-related attributes are only applicable to individuals who
are classified as being in employment.

For representative profiles that are not in employment, the attributes
**Industry** and **Distance travelled to work** are assigned
**"Does not apply"** to maintain logical consistency within the
representative population.

This constraint ensures that work-related attributes are only retained
for representative profiles with valid employment status.

### 5.1 Before Applying Constraint

In [8]:
EMPLOYED_STATUS = [
    "Economically active (excluding full-time students): In employment",
    "Economically active and a full-time student: In employment"
]

NON_EMPLOYED_MASK = (
    ~population_pool["Economic Activity Status"].isin(
        EMPLOYED_STATUS
    )
)

### 5.2 Industry(Before)

In [9]:
print("Industry before applying employment-related constraints")

display(
    population_pool.loc[
        NON_EMPLOYED_MASK,
        "Industry"
    ].value_counts()
)

Industry before applying employment-related constraints


Industry
A Agriculture, forestry and fishing                                        878080
F Construction                                                             878080
O Public administration and defence; compulsory social security            878080
N Administrative and support service activities                            878080
H Transport and storage                                                    878080
E Water supply; sewerage, waste management and remediation activities      878080
P Education                                                                878080
Does not apply                                                             878080
M Professional, scientific and technical activities                        878080
I Accommodation and food service activities                                878080
D Electricity, gas, steam and air conditioning supply                      878080
Q Human health and social work activities                                  878080
J Infor

### 5.3 Distance(Before)

In [10]:
print("Distance travelled to work before applying employment-related constraints")

display(
    population_pool.loc[
        NON_EMPLOYED_MASK,
        "Distance travelled to work"
    ].value_counts()
)

Distance travelled to work before applying employment-related constraints


Distance travelled to work
10km to less than 30km                                                            2085440
30km to less than 60km                                                            2085440
5km to less than 10km                                                             2085440
60km and over                                                                     2085440
Does not apply                                                                    2085440
Less than 5km                                                                     2085440
Works mainly at an offshore installation, in no fixed place, or outside the UK    2085440
Works mainly from home                                                            2085440
Name: count, dtype: int64

### 5.4 Number of inconsistencies identified

In [11]:
industry_inconsistent = (
    NON_EMPLOYED_MASK &
    (
        population_pool["Industry"]
        != "Does not apply"
    )
)

distance_inconsistent = (
    NON_EMPLOYED_MASK &
    (
        population_pool["Distance travelled to work"]
        != "Does not apply"
    )
)

print(
    f"Inconsistent industry profiles identified: "
    f"{industry_inconsistent.sum():,}"
)

print(
    f"Inconsistent commuting distance profiles identified: "
    f"{distance_inconsistent.sum():,}"
)

Inconsistent industry profiles identified: 15,805,440
Inconsistent commuting distance profiles identified: 14,598,080


### 5.5 Apply Constraint

In [12]:
population_pool.loc[
    NON_EMPLOYED_MASK,
    "Industry"
] = "Does not apply"

population_pool.loc[
    NON_EMPLOYED_MASK,
    "Distance travelled to work"
] = "Does not apply"

### 5.6 After Applying Constraint

In [13]:
print("Industry after applying employment-related constraints")

display(
    population_pool.loc[
        NON_EMPLOYED_MASK,
        "Industry"
    ].value_counts()
)

Industry after applying employment-related constraints


Industry
Does not apply    16683520
Name: count, dtype: int64

In [14]:
print("Distance travelled to work after applying employment-related constraints")

display(
    population_pool.loc[
        NON_EMPLOYED_MASK,
        "Distance travelled to work"
    ].value_counts()
)

Distance travelled to work after applying employment-related constraints


Distance travelled to work
Does not apply    16683520
Name: count, dtype: int64

### 6. Export Representative Population

In [ ]:
population_pool.to_parquet(
    output_path / "representative_population.parquet",
    index=False
)